In [1]:
from pathlib import Path
import pymupdf

# Locate the project root.
PROJECT_ROOT = Path.cwd()

# Handle cases where Jupyter starts inside the notebooks folder.
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PDF_DIR = PROJECT_ROOT / "data" / "raw_pdfs"
PDF_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("PDF folder:", PDF_DIR)

Project root: c:\Users\USER\Desktop\Gen_AI\DocuTrace_Rag
PDF folder: c:\Users\USER\Desktop\Gen_AI\DocuTrace_Rag\data\raw_pdfs


In [2]:
demo_pdfs = {
    "data_pipeline_runbook.pdf": [
        """
DATA PIPELINE RUNBOOK

Pipeline: CUSTOMER_DAILY_LOAD
Schedule: Daily at 02:00 UTC.
Upstream dependency: CUSTOMER_RAW_INGEST.

Normal operation:
1. Confirm the upstream ingestion completed successfully.
2. Confirm the expected business-date partition exists.
3. Start the downstream load after dependencies are healthy.

Owner: Data Operations.
""",
        """
DATA PIPELINE RUNBOOK - INCIDENT RESPONSE

Symptom: CUSTOMER_DAILY_LOAD failed because an upstream
partition is missing.

Investigation:
1. Check the upstream job status and logs.
2. Confirm the expected business date and partition format.
3. Verify the upstream ingestion completed.

Recovery:
Retry the downstream job only after the upstream partition
is available.

Safety: Do not overwrite production partitions during
investigation.
"""
    ],
    "hive_troubleshooting_guide.pdf": [
        """
HIVE TROUBLESHOOTING GUIDE

Symptom: A Hive query returns zero rows for an expected
business date.

Checks:
1. Confirm the database and table name.
2. Run SHOW PARTITIONS on the table.
3. Check the partition column and date format.
4. Verify that the upstream load completed successfully.

Important: A query may return no rows when it targets
the wrong partition.
""",
        """
SQL VALIDATION CHECKLIST

Before comparing source and target data:
1. Confirm filters and date ranges match.
2. Check total row counts.
3. Compare null counts for critical columns.
4. Compare aggregates such as SUM(amount), when appropriate.
5. Sample mismatched keys and inspect details.

Record the query, environment, execution date, and result.
Use read-only checks during investigation.
"""
    ]
}

for filename, page_texts in demo_pdfs.items():
    pdf_path = PDF_DIR / filename

    # Replace only our own generated training PDFs on reruns.
    if pdf_path.exists():
        pdf_path.unlink()

    with pymupdf.open() as pdf:
        for text in page_texts:
            page = pdf.new_page()

            page.insert_textbox(
                pymupdf.Rect(72, 72, 540, 760),
                text.strip(),
                fontsize=11,
                fontname="helv"
            )

        pdf.save(str(pdf_path))

    print("Created:", pdf_path.name)

Created: data_pipeline_runbook.pdf
Created: hive_troubleshooting_guide.pdf


In [3]:
pdf_paths = sorted(PDF_DIR.glob("*.pdf"))

print(f"Found {len(pdf_paths)} PDF files:\n")

for path in pdf_paths:
    print(path.name)

Found 2 PDF files:

data_pipeline_runbook.pdf
hive_troubleshooting_guide.pdf


In [4]:
extracted_pages = []

for pdf_path in pdf_paths:
    with pymupdf.open(str(pdf_path)) as pdf:

        metadata = pdf.metadata or {}

        for page_index, page in enumerate(pdf):
            page_text = page.get_text("text").strip()

            record = {
                "source": pdf_path.name,
                "title": metadata.get("title") or pdf_path.stem,
                "page_number": page_index + 1,
                "text": page_text
            }

            extracted_pages.append(record)

print("Total pages extracted:", len(extracted_pages))

Total pages extracted: 4


In [5]:
for record in extracted_pages:
    print(
        f"\n--- {record['source']} | "
        f"Page {record['page_number']} ---"
    )

    print(record["text"][:500])

empty_pages = [
    (record["source"], record["page_number"])
    for record in extracted_pages
    if not record["text"]
]

if empty_pages:
    print("\nPages with no selectable text:", empty_pages)
else:
    print("\nAll pages contain extractable text.")


--- data_pipeline_runbook.pdf | Page 1 ---
DATA PIPELINE RUNBOOK
Pipeline: CUSTOMER_DAILY_LOAD
Schedule: Daily at 02:00 UTC.
Upstream dependency: CUSTOMER_RAW_INGEST.
Normal operation:
1. Confirm the upstream ingestion completed successfully.
2. Confirm the expected business-date partition exists.
3. Start the downstream load after dependencies are healthy.
Owner: Data Operations.

--- data_pipeline_runbook.pdf | Page 2 ---
DATA PIPELINE RUNBOOK - INCIDENT RESPONSE
Symptom: CUSTOMER_DAILY_LOAD failed because an upstream
partition is missing.
Investigation:
1. Check the upstream job status and logs.
2. Confirm the expected business date and partition format.
3. Verify the upstream ingestion completed.
Recovery:
Retry the downstream job only after the upstream partition
is available.
Safety: Do not overwrite production partitions during
investigation.

--- hive_troubleshooting_guide.pdf | Page 1 ---
HIVE TROUBLESHOOTING GUIDE
Symptom: A Hive query returns zero rows for an expected
busin

In [6]:
import json

# Save the extracted page records for reuse.
extracted_path = (
    PROJECT_ROOT / "data" / "processed" / "extracted_pages.json"
)

extracted_path.parent.mkdir(parents=True, exist_ok=True)

with extracted_path.open("w", encoding="utf-8") as file:
    json.dump(
        extracted_pages,
        file,
        ensure_ascii=False,
        indent=2
    )

print(f"Saved {len(extracted_pages)} page records.")
print("File location:", extracted_path)

Saved 4 page records.
File location: c:\Users\USER\Desktop\Gen_AI\DocuTrace_Rag\data\processed\extracted_pages.json
